In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from xgboost import XGBRegressor

import pickle

from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor

In [ ]:
df = pd.read_csv("recommender_model_dataset.csv")

print(df.shape)
display(df.head())
print(df.columns.tolist())

In [ ]:
print("Saisons:")
print(df["season"].value_counts().sort_index())

print("\nFehlende Werte:")
display(df.isna().mean().sort_values(ascending=False).head(20))

print("\nTarget Summary:")
print(df["ziel_rating_avg_naechste_saison"].describe())

In [ ]:
target_col = "ziel_rating_avg_naechste_saison"

drop_cols = [
    target_col,
    # player_id nicht als Feature verwenden
    "player_id"
]

X = df.drop(columns=drop_cols).copy()
y = df[target_col].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
display(X.head())

In [ ]:
train_mask = df["season"] <= 2023
test_mask = df["season"] == 2024

X_train = X.loc[train_mask].copy()
y_train = y.loc[train_mask].copy()

X_test = X.loc[test_mask].copy()
y_test = y.loc[test_mask].copy()

season_train = df.loc[train_mask, "season"].copy()
season_test = df.loc[test_mask, "season"].copy()

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain seasons:")
print(season_train.value_counts().sort_index())

print("\nTest seasons:")
print(season_test.value_counts().sort_index())

In [ ]:
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)

baseline_preds = baseline.predict(X_test)

baseline_mae = mean_absolute_error(y_test, baseline_preds)
baseline_rmse = root_mean_squared_error(y_test, baseline_preds)
baseline_r2 = r2_score(y_test, baseline_preds)

print("Baseline Performance")
print(f"MAE :  {baseline_mae:.4f}")
print(f"RMSE:  {baseline_rmse:.4f}")
print(f"R²  :  {baseline_r2:.4f}")

In [ ]:
unique_train_seasons = sorted(season_train.unique())
print("Train seasons for CV:", unique_train_seasons)

cv = GroupKFold(n_splits=len(unique_train_seasons))
groups = season_train

In [ ]:
xgb = XGBRegressor(
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

In [ ]:
param_dist = {
    "n_estimators": [100, 200, 300, 500, 800],
    "max_depth": [2, 3, 4, 5, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05, 0.1, 0.15],
    "subsample": [0.6, 0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.6, 0.7, 0.8, 0.9, 1.0],
    "min_child_weight": [1, 2, 3, 5, 7],
    "gamma": [0, 0.1, 0.3, 0.5, 1.0],
    "reg_alpha": [0, 0.01, 0.1, 1.0, 5.0],
    "reg_lambda": [0.5, 1.0, 2.0, 5.0, 10.0]
}

In [ ]:
random_search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_dist,
    n_iter=40,
    scoring="neg_mean_absolute_error",
    n_jobs=-1,
    cv=cv,
    verbose=2,
    random_state=42,
    refit=True,
    return_train_score=True
)

random_search.fit(X_train, y_train, groups=groups)

In [ ]:
print("Best score (CV, neg MAE):", random_search.best_score_)
print("Best params:")
print(random_search.best_params_)

In [ ]:
cv_results = pd.DataFrame(random_search.cv_results_)
cv_results = cv_results.sort_values("rank_test_score")

cols_to_show = [
    "rank_test_score",
    "mean_test_score",
    "std_test_score",
    "mean_train_score",
    "param_n_estimators",
    "param_max_depth",
    "param_learning_rate",
    "param_subsample",
    "param_colsample_bytree",
    "param_min_child_weight",
    "param_gamma",
    "param_reg_alpha",
    "param_reg_lambda",
]

display(cv_results[cols_to_show].head(10))

In [ ]:
best_model = random_search.best_estimator_

test_preds = best_model.predict(X_test)

test_mae = mean_absolute_error(y_test, test_preds)
test_rmse = root_mean_squared_error(y_test, test_preds)
test_r2 = r2_score(y_test, test_preds)

print("XGBoost Test Performance")
print(f"MAE :  {test_mae:.4f}")
print(f"RMSE:  {test_rmse:.4f}")
print(f"R²  :  {test_r2:.4f}")

In [ ]:
comparison = pd.DataFrame({
    "Modell": ["Baseline", "XGBoost"],
    "MAE": [baseline_mae, test_mae],
    "RMSE": [baseline_rmse, test_rmse],
    "R2": [baseline_r2, test_r2]
})

display(comparison)

In [ ]:
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": best_model.feature_importances_
}).sort_values("importance", ascending=False)

display(feature_importance.head(20))

In [ ]:
pred_df = df.loc[test_mask, ["player_id", "season"]].copy()
pred_df["y_true"] = y_test.values
pred_df["y_pred"] = test_preds
pred_df["abs_error"] = np.abs(pred_df["y_true"] - pred_df["y_pred"])

display(pred_df.sort_values("abs_error").head(20))
display(pred_df.sort_values("abs_error", ascending=False).head(20))

In [ ]:
with open("recommender_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

print("Saved as: recommender_model.pkl")